# 401 · Lab: mini Protobuf subset encoder/decoder

Companion to [Lab: mini Protobuf encoder](https://leo-gan.github.io/GLD.SerializerBenchmark/theory/401/lab-mini-protobuf-encoder/).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leo-gan/GLD.SerializerBenchmark/blob/master/docs/theory/notebooks/401/lab_mini_protobuf_encoder.ipynb)

**Goal:** implement MiniUser encode/decode, match goldens G1–G5, skip unknowns, fail closed on truncated input, and cross-check with `google.protobuf` when available.

**Why this lab:** building a thin subset forces wire rules into muscle memory—without pretending to replace production runtimes.

**How to use:** study the reference codec, run done-when checks, then official-parser cross-check. Prefer re-implementing yourself in a fresh notebook/section.

**Expect:** all goldens match; unknown field skipped; round-trips hold; bounds cases raise `WireError`; optional `google.protobuf` parse of your bytes succeeds.

This is a **teaching subset**, not production Protobuf and not the suite fixture schema.

> **Honesty banner:** numbers and timings in these notebooks are **illustrative**. Suite [Results](https://leo-gan.github.io/GLD.SerializerBenchmark/) own harness truth for this project. Notebooks teach mechanisms, not leaderboards.


## Setup

Stdlib is enough for the subset codec. Optional: `protobuf` for the official-parser check.

**Why:** goldens do not require protoc; the oracle is a confidence boost.

**How:** detect `google.protobuf`; continue even if missing.

**Expect:** print whether the oracle is available.


In [ ]:
# Optional oracle (skip cells that need it if install fails)
try:
    from google import protobuf as _pb  # noqa: F401
    import google.protobuf as google_protobuf

    HAS_PROTOBUF = True
    print("google.protobuf available:", google_protobuf.__version__)
except ImportError:
    HAS_PROTOBUF = False
    print(
        "google.protobuf not installed — golden/bounds tests still run; "
        "install with: pip install protobuf"
    )



In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple


class WireError(Exception):
    """Bounds / wire failures (never silent truncate)."""


def hex_bytes(b: bytes) -> str:
    return " ".join(f"{x:02x}" for x in b)



## Teaching schema

```protobuf
syntax = "proto3";
message MiniUser {
  uint32 id = 1;
  string name = 2;
  MiniUser manager = 3;
  repeated uint32 tags = 4;  // unpacked
}
```

**In scope:** uint32 varint, string, nested message, unpacked repeated, skip unknown varint/LEN, proto3 omit defaults, reject truncated/overlong.

**Out of scope:** packed repeated, zigzag, maps, oneofs, full production hardening.

**Why this subset:** enough to exercise keys, varints, LEN, nesting, repeated, and skip—small enough to finish in one sitting.

**Why it matters:** labeling omissions is part of subset honesty (401 course rule).


## Reference implementation (study, then re-implement yourself)

**Why:** a working model clarifies the article pseudocode; the learning comes from rewriting it.

**How:** read encode/decode carefully; note proto3 omit-default and unknown skip paths.

**Expect:** definitions only until the next checks cell runs.

**Why it matters:** production systems use codegen runtimes—this trains **judgment** when those runtimes misbehave or when you read raw bytes.


In [ ]:
def encode_varint(u: int) -> bytes:
    if u < 0:
        raise ValueError("unsigned only")
    out = bytearray()
    while u > 0x7F:
        out.append((u & 0x7F) | 0x80)
        u >>= 7
    out.append(u & 0x7F)
    return bytes(out)


def decode_varint(buf: bytes, i: int = 0) -> Tuple[int, int]:
    value = 0
    shift = 0
    nread = 0
    while True:
        if i >= len(buf):
            raise WireError("truncated varint")
        b = buf[i]
        i += 1
        nread += 1
        if nread > 10:
            raise WireError("overlong varint")
        value |= (b & 0x7F) << shift
        if (b & 0x80) == 0:
            return value, i
        shift += 7


def encode_key(field_number: int, wire_type: int) -> bytes:
    return encode_varint((field_number << 3) | wire_type)


def require(buf: bytes, i: int, n: int) -> Tuple[bytes, int]:
    if n < 0 or i + n > len(buf):
        raise WireError("truncated payload")
    return buf[i : i + n], i + n


@dataclass
class MiniUser:
    id: int = 0
    name: str = ""
    manager: Optional["MiniUser"] = None
    tags: List[int] = field(default_factory=list)

    def logical(self) -> tuple:
        return (
            self.id,
            self.name,
            None if self.manager is None else self.manager.logical(),
            list(self.tags),
        )


def encode_mini_user(u: MiniUser) -> bytes:
    out = bytearray()
    if u.id != 0:
        out += encode_key(1, 0) + encode_varint(u.id)
    if u.name:
        b = u.name.encode("utf-8")
        out += encode_key(2, 2) + encode_varint(len(b)) + b
    if u.manager is not None:
        inner = encode_mini_user(u.manager)
        out += encode_key(3, 2) + encode_varint(len(inner)) + inner
    for t in u.tags:
        out += encode_key(4, 0) + encode_varint(t)
    return bytes(out)


def decode_mini_user(buf: bytes) -> MiniUser:
    i = 0
    user = MiniUser()
    while i < len(buf):
        key, i = decode_varint(buf, i)
        fn, wt = key >> 3, key & 7
        if wt == 0:
            v, i = decode_varint(buf, i)
            if fn == 1:
                user.id = v
            elif fn == 4:
                user.tags.append(v)
            # else: unknown varint already consumed
        elif wt == 2:
            n, i = decode_varint(buf, i)
            payload, i = require(buf, i, n)
            if fn == 2:
                user.name = payload.decode("utf-8")
            elif fn == 3:
                user.manager = decode_mini_user(payload)
            # else: skip unknown LEN
        elif wt == 1:
            _, i = require(buf, i, 8)
        elif wt == 5:
            _, i = require(buf, i, 4)
        else:
            raise WireError(f"unsupported wire type {wt}")
    return user



## Done-when checks

**Why:** the lab article’s “done when” list is the acceptance test—not vibes.

**How:** run goldens G1–G5, unknown skip (`G1 + 28 63`), round-trips, and three bounds failures.

**Expect:** every check prints `OK …`; bounds cases raise `WireError` (never silent truncate).

**Why it matters:** decoders that “usually work” on happy paths still fail production and security review without fail-closed bounds.


In [ ]:
GOLDENS = {
    "G1": (MiniUser(id=1, name="Ada"), "08 01 12 03 41 64 61"),
    "G2": (MiniUser(), ""),
    "G3": (MiniUser(id=300), "08 ac 02"),
    "G4": (MiniUser(tags=[1, 2]), "20 01 20 02"),
    "G5": (MiniUser(manager=MiniUser(id=2)), "1a 02 08 02"),
}

print("=== 1. Encode goldens G1–G5 ===")
for label, (user, hx) in GOLDENS.items():
    raw = encode_mini_user(user)
    exp = bytes.fromhex(hx.replace(" ", "")) if hx else b""
    assert raw == exp, f"{label}: {hex_bytes(raw)} != {hex_bytes(exp)}"
    print(f"OK {label}: {hex_bytes(raw) if raw else '<empty>'}")

print("\n=== 2. Unknown skip: G1 + 28 63 (field 5 = 99) ===")
g1 = bytes.fromhex("08011203416461")
with_unknown = g1 + bytes.fromhex("28 63")
d = decode_mini_user(with_unknown)
assert d.logical() == (1, "Ada", None, [])
print("OK unknown skipped; logical", d.logical())

print("\n=== 3. Round-trip ===")
samples = [
    MiniUser(id=1, name="Ada"),
    MiniUser(id=300, name="Grace", tags=[1, 2, 3]),
    MiniUser(id=7, manager=MiniUser(id=2, name="Boss"), tags=[9]),
    MiniUser(),
]
for s in samples:
    back = decode_mini_user(encode_mini_user(s))
    assert back.logical() == s.logical(), (s.logical(), back.logical())
    print("OK round-trip", s.logical())

print("\n=== 4. Bounds (must raise WireError) ===")
cases = [
    ("LEN n=10 but 2 bytes left", bytes.fromhex("12 0a 41 42")),
    ("truncated varint", bytes.fromhex("80")),
    ("fixed64 short", encode_key(9, 1) + b"\x00\x01"),
]
for name, blob in cases:
    try:
        decode_mini_user(blob)
        raise AssertionError(f"expected WireError for {name}")
    except WireError as e:
        print(f"OK {name}: {e}")

print("\nAll subset checks passed.")



## Official parser cross-check (`google.protobuf`)

**Why:** matching your own round-trip is necessary but not sufficient—an official parser is an independent oracle.

**How:** build MiniUser via `FileDescriptorProto` (no `protoc` binary); `ParseFromString` on your G1/G5 encodings and golden hex.

**Expect:** SKIP if protobuf missing; otherwise `id`/`name`/nested manager match.

**Why it matters:** golden bytes + official parse beat “looks right in hex.” Cross-language appendices (Go/Rust) use the same vectors.


In [ ]:
if not HAS_PROTOBUF:
    print("SKIP official oracle — pip install protobuf")
else:
    from google.protobuf import descriptor_pb2, descriptor_pool, message_factory

    file_proto = descriptor_pb2.FileDescriptorProto()
    file_proto.name = "mini.proto"
    file_proto.package = "lab"
    file_proto.syntax = "proto3"

    msg = file_proto.message_type.add()
    msg.name = "MiniUser"

    f = msg.field.add()
    f.name, f.number, f.label = "id", 1, descriptor_pb2.FieldDescriptorProto.LABEL_OPTIONAL
    f.type = descriptor_pb2.FieldDescriptorProto.TYPE_UINT32

    f = msg.field.add()
    f.name, f.number, f.label = "name", 2, descriptor_pb2.FieldDescriptorProto.LABEL_OPTIONAL
    f.type = descriptor_pb2.FieldDescriptorProto.TYPE_STRING

    f = msg.field.add()
    f.name, f.number, f.label = "manager", 3, descriptor_pb2.FieldDescriptorProto.LABEL_OPTIONAL
    f.type = descriptor_pb2.FieldDescriptorProto.TYPE_MESSAGE
    f.type_name = ".lab.MiniUser"

    f = msg.field.add()
    f.name, f.number, f.label = "tags", 4, descriptor_pb2.FieldDescriptorProto.LABEL_REPEATED
    f.type = descriptor_pb2.FieldDescriptorProto.TYPE_UINT32

    pool = descriptor_pool.DescriptorPool()
    pool.Add(file_proto)
    desc = pool.FindMessageTypeByName("lab.MiniUser")
    MiniUserPb = message_factory.GetMessageClass(desc)

    raw = encode_mini_user(MiniUser(id=1, name="Ada"))
    m = MiniUserPb()
    m.ParseFromString(raw)
    assert m.id == 1 and m.name == "Ada"
    print("OK official ParseFromString on our G1 encode:", m.id, m.name)

    golden = bytes.fromhex("08011203416461")
    m2 = MiniUserPb()
    m2.ParseFromString(golden)
    assert m2.id == 1 and m2.name == "Ada"
    print("OK official parse of golden G1 hex")

    g5 = encode_mini_user(MiniUser(manager=MiniUser(id=2)))
    m3 = MiniUserPb()
    m3.ParseFromString(g5)
    assert m3.manager.id == 2
    print("OK official parse nested manager id=2")



## Stretch ideas

- Packed `repeated uint32` (single LEN, concatenated varints)
- Reject trailing garbage inside a nested LEN (strict consume)
- Same goldens in another language — see `docs/theory/notebooks/companions/`

**Why stretch:** packed repeated and strict nested consume show up in real interop and fuzzing.

## Next

- [Python: google.protobuf path](../../401/protobuf-python.md)
- [Cross-language fidelity](../../401/protobuf-cross-language-fidelity.md)
